# Final Model Evaluation

Lock the selected boosting model and evaluate once on the untouched May 23–June 6 test period.


## Load the Locked Candidate and Fit on Train plus Validation

Use the selected 31-leaf settings and all labels available before the test period.


In [1]:
from pathlib import Path
import json
import time
import joblib
import numpy as np
import pandas as pd
import db_dtypes
from google.cloud import bigquery
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss, log_loss
from sklearn.pipeline import Pipeline
from threadpoolctl import threadpool_limits

project_root = Path.cwd().resolve()
if not (project_root / "dbt_promo_abuse").is_dir():
    project_root = project_root.parent
comparison_dir = project_root / "data" / "processed" / "model_comparison"
train_df = pd.read_parquet(comparison_dir / "train.parquet")
validation_df = pd.read_parquet(comparison_dir / "validation.parquet")
feature_columns = json.loads((comparison_dir / "feature_columns.json").read_text())
assert len(feature_columns) == len(set(feature_columns)) == 31
assert not set(feature_columns) & {"user_id", "prediction_date", "abuse_next_7d", "dataset_split"}
assert train_df["dataset_split"].eq("train").all()
assert validation_df["dataset_split"].eq("validation").all()
assert len(train_df) == 483431 and train_df["abuse_next_7d"].sum() == 5552
assert len(validation_df) == 102860 and validation_df["abuse_next_7d"].sum() == 1149

fit_df = pd.concat([train_df, validation_df], ignore_index=True)
X_fit = fit_df[feature_columns].astype("float64")
y_fit = fit_df["abuse_next_7d"].astype("int64")
recency_columns = [c for c in feature_columns if c.startswith("days_since_")]
ratio_columns = ["offline_redemption_rate_30d"]
complete_columns = [c for c in feature_columns if c not in recency_columns + ratio_columns]
assert not np.isinf(X_fit.to_numpy()).any()
assert not X_fit[complete_columns].isna().any().any()
preprocessing = ColumnTransformer([
    ("recency", SimpleImputer(strategy="constant", fill_value=31), recency_columns),
    ("ratio", SimpleImputer(strategy="constant", fill_value=0, add_indicator=True), ratio_columns),
    ("complete", "passthrough", complete_columns),
], remainder="drop")
selected_model = Pipeline([
    ("preprocess", preprocessing),
    ("model", HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_iter=200,
        max_leaf_nodes=31,
        min_samples_leaf=30,
        l2_regularization=1.0,
        early_stopping=False,
        class_weight=None,
        random_state=42,
    )),
])
started = time.perf_counter()
with threadpool_limits(limits=2):
    selected_model.fit(X_fit, y_fit)
print(f"Fitted locked candidate on {len(X_fit):,} train+validation rows ({y_fit.sum():,} positives) in {time.perf_counter()-started:.1f}s.")


Fitted locked candidate on 586,291 train+validation rows (6,701 positives) in 13.4s.


## Load Test Period Once

Verify the held-out schedule and inspect test outcomes only for final evaluation.


In [2]:
PROJECT_ID = "alibaba-promo-abuse-1790454859"
client = bigquery.Client(project=PROJECT_ID)
test_sql = f"""
SELECT *
FROM `{PROJECT_ID}.promo_abuse_features.user_modeling_dataset`
WHERE dataset_split = 'test'
  AND prediction_date BETWEEN DATE '2016-05-23' AND DATE '2016-06-06'
"""
max_bytes_billed = 1_000_000_000
test_dry_run = client.query(
    test_sql,
    job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False),
)
print(f"Estimated test scan: {test_dry_run.total_bytes_processed / 1_000_000:.1f} MB")
if test_dry_run.total_bytes_processed > max_bytes_billed:
    raise RuntimeError("Test query exceeds the 1 GB billing cap.")
test_job = client.query(
    test_sql,
    job_config=bigquery.QueryJobConfig(maximum_bytes_billed=max_bytes_billed),
)
test_df = test_job.result().to_dataframe(create_bqstorage_client=False)
assert test_df["dataset_split"].eq("test").all()
assert not test_df.duplicated(["user_id", "prediction_date"]).any()
assert test_df[["user_id", "prediction_date"]].notna().all().all()
assert test_df["abuse_next_7d"].isin([0, 1]).all()
assert set(pd.to_datetime(test_df["prediction_date"])) == set(
    pd.date_range("2016-05-23", "2016-06-06", freq="7D")
)
X_test = test_df[feature_columns].astype("float64")
y_test = test_df["abuse_next_7d"].astype("int64")
assert not np.isinf(X_test.to_numpy()).any()
assert not X_test[complete_columns].isna().any().any()
print(f"Test rows: {len(test_df):,}; positives: {y_test.sum():,}; prevalence: {y_test.mean():.3%}")


Estimated test scan: 39.9 MB
Test rows: 137,183; positives: 1,549; prevalence: 1.129%


## Final Test Metrics and Review Capacity

Report ranking, probability quality, and user-week outcomes at the fixed weekly review capacities.


In [3]:
test_scores = selected_model.predict_proba(X_test)[:, 1]
test_predictions = test_df[["user_id", "prediction_date", "abuse_next_7d"]].copy()
test_predictions["risk_score"] = test_scores
constant_test_scores = np.full(len(y_test), y_fit.mean())
test_metrics = pd.DataFrame([
    {
        "model": "Selected histogram gradient boosting",
        "average_precision": average_precision_score(y_test, test_scores),
        "roc_auc": roc_auc_score(y_test, test_scores),
        "brier_score": brier_score_loss(y_test, test_scores),
        "log_loss": log_loss(y_test, test_scores, labels=[0, 1]),
    },
    {
        "model": "Constant train+validation prevalence",
        "average_precision": average_precision_score(y_test, constant_test_scores),
        "roc_auc": roc_auc_score(y_test, constant_test_scores),
        "brier_score": brier_score_loss(y_test, constant_test_scores),
        "log_loss": log_loss(y_test, constant_test_scores, labels=[0, 1]),
    },
])
display(test_metrics.round(5))
weekly_rows = []
for date, week in test_predictions.groupby("prediction_date"):
    ranked = week.sort_values(["risk_score", "user_id"], ascending=[False, True], kind="mergesort")
    positives = int(week["abuse_next_7d"].sum())
    for capacity in [100, 250, 500]:
        selected = ranked.head(capacity)
        tp = int(selected["abuse_next_7d"].sum())
        weekly_rows.append({
            "prediction_date": date, "capacity": capacity, "reviewed": len(selected),
            "positives": positives, "true_positives": tp,
            "false_positives": len(selected)-tp,
            "precision_at_k": tp/len(selected),
            "recall_at_k": tp/positives if positives else np.nan,
        })
test_weekly_capacity = pd.DataFrame(weekly_rows)
test_capacity_summary = test_weekly_capacity.groupby("capacity").agg(
    reviewed=("reviewed", "sum"), positives=("positives", "sum"),
    true_positives=("true_positives", "sum"), false_positives=("false_positives", "sum"),
)
test_capacity_summary["precision_at_k"] = test_capacity_summary["true_positives"] / test_capacity_summary["reviewed"]
test_capacity_summary["recall_at_k"] = test_capacity_summary["true_positives"] / test_capacity_summary["positives"].replace(0, np.nan)
display(test_weekly_capacity.round(4))
display(test_capacity_summary.round(4))
print("These are final held-out results for the controlled evaluation scenarios.")


,model,average_precision,roc_auc,brier_score,log_loss
0,Selected histogram gradient boosting,0.39763,0.95952,0.00866,0.03508
1,Constant train+validation prevalence,0.01129,0.50000,0.01116,0.06186


,prediction_date,capacity,reviewed,positives,true_positives,false_positives,precision_at_k,recall_at_k
0,2016-05-23,100,100,473,82,18,0.820,0.1734
1,2016-05-23,250,250,473,139,111,0.556,0.2939
2,2016-05-23,500,500,473,191,309,0.382,0.4038
3,2016-05-30,100,100,519,85,15,0.850,0.1638
4,2016-05-30,250,250,519,140,110,0.560,0.2697
5,2016-05-30,500,500,519,203,297,0.406,0.3911
6,2016-06-06,100,100,557,75,25,0.750,0.1346
7,2016-06-06,250,250,557,154,96,0.616,0.2765
8,2016-06-06,500,500,557,214,286,0.428,0.3842


,reviewed,positives,true_positives,false_positives,precision_at_k,recall_at_k
capacity,,,,,,
100,300,1549,242,58,0.8067,0.1562
250,750,1549,433,317,0.5773,0.2795
500,1500,1549,608,892,0.4053,0.3925


These are final held-out results for the controlled evaluation scenarios.


## Save Final Candidate and Test Scores

Save the fitted pipeline and ranked test outputs locally for reproducibility.


In [4]:
final_dir = project_root / "data" / "processed" / "model_comparison"
joblib.dump(selected_model, final_dir / "selected_gradient_boosting.joblib")
test_predictions.sort_values(
    ["prediction_date", "risk_score", "user_id"],
    ascending=[True, False, True], kind="mergesort",
).to_parquet(final_dir / "final_test_scores.parquet", index=False)
print("Saved final model and test scores to", final_dir)


Saved final model and test scores to /Users/andrewlu/Desktop/alibaba-promo-abuse/data/processed/model_comparison
